# Phase 4 - Court Judgment Prediction (lightweight Phase-1 model)

**Task:** predict a binary outcome (0/1) for each court case from its full text; scored by macro-F1.

**What you need on disk (nothing else - the model code is embedded below):**
1. your Phase-1 lightweight checkpoint `lightweight_final.pt` (must contain `model_state` and `config`)
2. the Phase-1 tokenizer pickle `legal_bpe_v3.pkl` (the `.pt` does **not** contain it)
3. a Hugging Face token (Kaggle secret `hf_token`)

**Approach (hierarchical, long-document):** a judgment is cut into `CH`-token chunks; up to `K_CHUNKS` of them are selected (`SELECT` = head / tail / headtail / spread); the lightweight LM (4 shared blocks run twice, causal 128-token local attention) encodes every chunk into a vector; a document-level aggregator (learned attention pooling, mean, or BiGRU) combines the chunk vectors; a linear layer gives the outcome. Class-weighted loss for the imbalance; an optional dev-tuned threshold as calibration.

**Experiments:** full fine-tuning, partial (layer-wise) fine-tuning, LoRA, **LoRMA+** (compulsory), bottleneck adapters, knowledge distillation (student = half the unique blocks), plus long-document ablations (which chunks, which aggregator). AdaLoRA is **not** included.

> The course requires all **four** Phase-1 architectures and >= 6 substantially different experiments (at least 5 different adaptation techniques, LoRMA and knowledge distillation among them). Hyperparameters are placeholders - choosing and justifying them is your job (course rules).

## 0. Config - edit the paths

In [ ]:
TOKENIZER_PATH = "/kaggle/input/YOUR_DATASET/legal_bpe_v3.pkl"      # <-- Phase-1 tokenizer
CKPT_PATH      = "/kaggle/input/YOUR_DATASET/lightweight_final.pt"       # <-- Phase-1 lightweight checkpoint
HF_REPO = "Exploration-Lab/CS-781-Capstone"
OUT = "/kaggle/working"

CH         = 512      # tokens per chunk (<= the lightweight model's max_seq_len of 1024)
STRIDE_CH  = 512      # chunk stride; < CH gives overlapping chunks
K_CHUNKS   = 8        # at most this many chunks of a document go through the model per step
SELECT     = "headtail"   # which chunks: head | tail | headtail | spread   (the long-document / section experiment)
CAP_CHARS  = 60000    # very long judgments: keep this many characters from each end (bounds tokenisation time)

## 1. Imports and Hugging Face login

In [ ]:
import copy, json, math, os, pickle, random, re, time, zipfile
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "| torch", torch.__version__)

from collections import Counter

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("hf_token")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")        # Colab / local: export HF_TOKEN yourself
from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN); print("HF login ok")
else:
    print("no HF token found - dataset download may fail")

## 2. Phase-1 tokenizer (class copied from your Phase-1 notebook)

In [ ]:
class LegalBPETokenizer:
    """Phase-1 tokenizer, copied unchanged from your Phase-1 notebook (needs legal_bpe_v3.pkl)."""

    def __init__(self, model_path):
        with open(model_path, "rb") as f:
            model = pickle.load(f)
        self.vocab_size = model["vocab_size"]
        self.merges = model["merges"]                     # [((a, b), new_id), ...]
        self.lowercase = model.get("lowercase", False)
        self.special_tokens = model.get("special_tokens", [])
        self.pattern = re.compile(model["pattern"])
        self.merge_rank = {tuple(pair): rank for rank, (pair, _) in enumerate(self.merges)}
        self.token_bytes = {i: bytes([i]) for i in range(256)}
        for (l, r), new_id in self.merges:
            self.token_bytes[new_id] = self.token_bytes[l] + self.token_bytes[r]
        base = 256 + len(self.merges)
        self.special_to_id = {t: base + i for i, t in enumerate(self.special_tokens)}
        self.id_to_special = {v: k for k, v in self.special_to_id.items()}
        self.eos_id = self.special_to_id.get("<|endoftext|>")
        self._special_re = (re.compile("(" + "|".join(re.escape(t) for t in self.special_tokens) + ")")
                            if self.special_tokens else None)
        self._cache = {}

    def _encode_piece(self, piece_bytes):
        cached = self._cache.get(piece_bytes)
        if cached is not None:
            return cached
        ids = list(piece_bytes)
        while len(ids) > 1:
            best, best_rank = None, None
            for p in zip(ids, ids[1:]):
                r = self.merge_rank.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = p, r
            if best is None:
                break
            new_id = 256 + best_rank
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == best[0] and ids[i + 1] == best[1]:
                    out.append(new_id); i += 2
                else:
                    out.append(ids[i]); i += 1
            ids = out
        if len(self._cache) < 500_000:
            self._cache[piece_bytes] = ids
        return ids

    def _encode_ordinary(self, text):
        if self.lowercase:
            text = text.lower()
        out = []
        for piece in self.pattern.findall(text):
            out.extend(self._encode_piece(piece.encode("utf-8")))
        return out

    def encode(self, text, add_eos=False):
        if self._special_re is not None:
            ids = []
            for part in self._special_re.split(text):
                if part in self.special_to_id:
                    ids.append(self.special_to_id[part])
                elif part:
                    ids.extend(self._encode_ordinary(part))
        else:
            ids = self._encode_ordinary(text)
        if add_eos:
            ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        chunks = []
        for token_id in ids:
            if token_id in self.id_to_special:
                chunks.append(self.id_to_special[token_id].encode("utf-8"))
            else:
                chunks.append(self.token_bytes[token_id])
        return b"".join(chunks).decode("utf-8", errors="replace")


tokenizer = LegalBPETokenizer(TOKENIZER_PATH)
print("vocab size:", tokenizer.vocab_size, "| lowercase:", tokenizer.lowercase, "| eos id:", tokenizer.eos_id)
print([tokenizer.decode([i]) for i in tokenizer.encode("The appellant filed this appeal.")])

## 3. Phase-1 lightweight model code (copied unchanged) and checkpoint loading

In [ ]:
# ---- lightweight_layers.py (Phase 1, unchanged) ----
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        rms = x.float().pow(2).mean(dim=-1, keepdim=True)
        x = x * torch.rsqrt(rms + self.eps)
        return x * self.weight.to(x.dtype)


def get_alibi_slopes(n_heads):
    def power_of_2_slopes(n):
        start = 2 ** (-2 ** -(math.log2(n) - 3))
        ratio = start
        return torch.tensor([start * ratio ** i for i in range(n)], dtype=torch.float32)
    if math.log2(n_heads).is_integer():
        return power_of_2_slopes(n_heads)
    closest_power = 2 ** int(math.floor(math.log2(n_heads)))
    extra = n_heads - closest_power
    slopes = power_of_2_slopes(closest_power)
    extra_slopes = get_alibi_slopes(2 * closest_power)[0::2]
    return torch.cat([slopes, extra_slopes[:extra]])


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_model, d_ff, bias=False)
        self.w3 = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.w3(F.silu(self.w1(x)) * self.w2(x)))


class LocalGQAAttention(nn.Module):
    """Causal sliding-window grouped-query attention with ALiBi."""
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm=True):
        super().__init__()
        assert d_model % n_q_heads == 0 and n_q_heads % n_kv_heads == 0
        self.d_model = d_model
        self.n_q_heads, self.n_kv_heads = n_q_heads, n_kv_heads
        self.groups = n_q_heads // n_kv_heads
        self.d_head = d_model // n_q_heads
        self.window_size = min(window_size, max_seq_len)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.qk_norm = qk_norm
        if qk_norm:
            self.q_norm = RMSNorm(self.d_head)
            self.k_norm = RMSNorm(self.d_head)
        self.dropout = dropout
        self.register_buffer("alibi_slopes", get_alibi_slopes(n_q_heads), persistent=False)
        distance = torch.arange(self.window_size - 1, -1, -1, dtype=torch.float32)
        self.register_buffer("distance", distance, persistent=False)
        query_pos = torch.arange(max_seq_len).unsqueeze(1)
        window_pos = torch.arange(self.window_size).unsqueeze(0)
        valid = window_pos >= (self.window_size - 1 - query_pos)
        self.register_buffer("valid_mask", valid, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_q_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.qk_norm:
            q, k = self.q_norm(q), self.k_norm(k)
        q = q.view(B, self.n_kv_heads, self.groups, T, self.d_head)
        W = self.window_size
        k_win = F.pad(k, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)   # (B, Hkv, T, W, D)
        v_win = F.pad(v, (0, 0, W - 1, 0)).unfold(2, W, 1).permute(0, 1, 2, 4, 3)
        scores = torch.einsum("bhgtd,bhtwd->bhgtw", q, k_win) / math.sqrt(self.d_head)
        slopes = self.alibi_slopes.view(1, self.n_kv_heads, self.groups, 1, 1)
        scores = scores - slopes * self.distance.view(1, 1, 1, 1, W)
        mask = self.valid_mask[:T, :W].view(1, 1, 1, T, W)
        scores = scores.masked_fill(~mask, float("-inf"))
        attn = F.softmax(scores.float(), dim=-1).to(scores.dtype)
        if self.training and self.dropout > 0:
            attn = F.dropout(attn, p=self.dropout)
        out = torch.einsum("bhgtw,bhtwd->bhgtd", attn, v_win)
        out = out.reshape(B, self.n_q_heads, T, self.d_head).transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class LightweightBlock(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, d_ff, dropout, qk_norm):
        super().__init__()
        self.ln1 = RMSNorm(d_model)
        self.attn = LocalGQAAttention(d_model, n_q_heads, n_kv_heads, max_seq_len, window_size, dropout, qk_norm)
        self.ln2 = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.ffn(self.ln2(x))


# ---- model_arch_lightweight.py (Phase 1, unchanged) ----
from dataclasses import dataclass

@dataclass
class LightweightConfig:
    vocab_size: int
    d_model: int = 1024
    n_q_heads: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4096
    max_seq_len: int = 1024
    window_size: int = 128
    dropout: float = 0.0
    qk_norm: bool = True
    n_unique_blocks: int = 4       # 4 unique blocks ...
    n_repeats: int = 2             # ... each executed twice: B1 B2 B3 B4 B1 B2 B3 B4


class LightweightLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([
            LightweightBlock(cfg.d_model, cfg.n_q_heads, cfg.n_kv_heads, cfg.max_seq_len, cfg.window_size,
                             cfg.d_ff, cfg.dropout, cfg.qk_norm)
            for _ in range(cfg.n_unique_blocks)])
        self.ln_f = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight            # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.tok_emb(x)
        for _ in range(self.cfg.n_repeats):
            for block in self.blocks:
                x = block(x)
        return self.lm_head(self.ln_f(x))

In [ ]:
def load_lm(path=CKPT_PATH, dropout=0.1):
    # weights_only=False: this is YOUR OWN checkpoint (config dict + tensors), not an untrusted file
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = LightweightConfig(**{k: v for k, v in ck["config"].items() if k in LightweightConfig.__dataclass_fields__})
    assert cfg.vocab_size == tokenizer.vocab_size, \
        f"checkpoint vocab {cfg.vocab_size} != tokenizer vocab {tokenizer.vocab_size}: wrong tokenizer .pkl?"
    cfg.dropout = dropout                                    # Phase 1 used 0.0; a little helps fine-tuning
    lm = LightweightLM(cfg)
    lm.load_state_dict(ck["model_state"])                    # alibi / mask buffers are non-persistent: rebuilt, not loaded
    return lm

_lm = load_lm(dropout=0.0)
print(_lm.cfg)
print(f"loaded Phase-1 lightweight LM: {sum(p.numel() for p in _lm.parameters())/1e6:.1f}M unique params "
      f"({_lm.cfg.n_unique_blocks} blocks x {_lm.cfg.n_repeats} repeats)")
assert _lm.cfg.max_seq_len >= CH, "CH must be <= max_seq_len"

## 4. Data
### 4.1 Peek at the raw JSON (verify the layout and the label key before trusting the parser)

In [ ]:
def hf_json(path):
    return json.load(open(hf_hub_download(HF_REPO, path, repo_type="dataset"), encoding="utf-8"))

for name, path in [("train", "Phase_4_CJPE/Train/train.json"), ("dev", "Phase_4_CJPE/Train/dev.json"),
                   ("test", "Phase_4_CJPE/Test/test.json")]:
    raw = hf_json(path)
    first = next(iter(raw.items())) if isinstance(raw, dict) else raw[0]
    if isinstance(first, dict):
        show = {k: (str(v)[:120] + "..." if len(str(v)) > 120 else v) for k, v in first.items()}
    else:
        show = str(first)[:300]
    print(name, type(raw).__name__, "len", len(raw), "\n   first:", show, "\n")

### 4.2 Parse train / dev / test, class balance

In [ ]:
LABEL_KEYS = ("label", "labels", "y", "outcome", "target", "judgment", "decision")

def parse_docs(raw):
    if isinstance(raw, dict):
        items = [(k, v) for k, v in raw.items()]
    else:
        items = [(d["id"], d) for d in raw]
    docs = []
    for did, d in items:
        if not isinstance(d, dict):
            d = {"text": d}
        lab = next((d[k] for k in LABEL_KEYS if k in d), None)
        docs.append(dict(id=d.get("id", did), text=d["text"], label=None if lab is None else int(lab)))
    return docs

train_docs = parse_docs(hf_json("Phase_4_CJPE/Train/train.json"))
dev_docs   = parse_docs(hf_json("Phase_4_CJPE/Train/dev.json"))
test_docs  = parse_docs(hf_json("Phase_4_CJPE/Test/test.json"))
for name, ds in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    labs = [d["label"] for d in ds]
    extra = f" | label counts {dict(Counter(labs))}" if labs[0] is not None else ""
    print(f"{name}: {len(ds)} docs{extra}")
assert all(d["label"] in (0, 1) for d in train_docs + dev_docs), "labels must be 0/1 - check the label key printed above"
assert len({str(d["id"]) for d in test_docs}) == len(test_docs), "duplicate test ids"
print("median chars/doc:", int(np.median([len(d["text"]) for d in train_docs])),
      "| max:", max(len(d["text"]) for d in train_docs))

### 4.3 Tokenise and chunk

In [ ]:
def chunk_ids(ids, ch=CH, stride=STRIDE_CH):
    starts = [0]
    while starts[-1] + ch < len(ids):
        starts.append(starts[-1] + stride)
    return [ids[s:s + ch] for s in starts]

def tokenise_doc(text):
    if len(text) > 2 * CAP_CHARS:
        text = text[:CAP_CHARS] + " " + text[-CAP_CHARS:]
    ids = tokenizer.encode(text)
    return ids if ids else [tokenizer.eos_id]

t0 = time.time()
for d in train_docs + dev_docs + test_docs:
    ids = tokenise_doc(d["text"])
    d["n_tok"] = len(ids)
    d["chunks"] = chunk_ids(ids)
print(f"tokenised in {time.time()-t0:.0f}s | tokens/doc: median {int(np.median([d['n_tok'] for d in train_docs]))}, "
      f"p90 {int(np.percentile([d['n_tok'] for d in train_docs], 90))} | chunks/doc: "
      f"median {int(np.median([len(d['chunks']) for d in train_docs]))}, max {max(len(d['chunks']) for d in train_docs)}")

### 4.4 Chunk selection and batching

In [ ]:
def select_idx(n, select=None, K=None):
    """Which of the n chunks of a document are used.  head = first K, tail = last K (judgments end with the outcome),
    headtail = first K/2 + last K/2, spread = K evenly spaced chunks over the whole document."""
    select, K = select or SELECT, K or K_CHUNKS
    if n <= K:
        return list(range(n))
    if select == "head":
        return list(range(K))
    if select == "tail":
        return list(range(n - K, n))
    if select == "headtail":
        h = K // 2
        return list(range(h)) + list(range(n - (K - h), n))
    if select == "spread":
        return sorted({round(i * (n - 1) / max(1, K - 1)) for i in range(K)})
    raise ValueError(select)

def collate(docs, select=None, K=None):
    """-> X (N,T) chunk tokens | lens (N,) | cnt (D,) chunks per doc | Y (D,) labels.  Chunks of all docs are stacked."""
    seqs, cnt = [], []
    for d in docs:
        idx = select_idx(len(d["chunks"]), select, K)
        seqs += [d["chunks"][i] for i in idx]; cnt.append(len(idx))
    T = max(len(s) for s in seqs)
    X = torch.zeros(len(seqs), T, dtype=torch.long)
    for i, s in enumerate(seqs):
        X[i, :len(s)] = torch.tensor(s)
    lens = torch.tensor([len(s) for s in seqs])
    Y = torch.tensor([d["label"] if d["label"] is not None else -1 for d in docs])
    return X, lens, torch.tensor(cnt), Y

print("example selections for a 20-chunk doc:", {s: select_idx(20, s) for s in ("head", "tail", "headtail", "spread")})

### 4.5 Checkpoint / tokenizer sanity check

In [ ]:
# Does checkpoint + tokenizer agree? LM loss on real legal text should be ~3-4 nats; ~10 means a wrong tokenizer pkl.
x = torch.tensor([train_docs[0]["chunks"][0]]); _lm.eval()
with torch.no_grad():
    lg = _lm(x)
print("Phase-1 LM loss on a train chunk:", round(F.cross_entropy(lg[0, :-1], x[0, 1:]).item(), 3), "| tokens:", x.size(1))
del _lm

## 5. Metric (macro-F1) and threshold calibration

In [ ]:
def binary_scores(gold, pred):
    """Macro precision / recall / F1 over the two classes (the competition metric is macro-F1)."""
    P, R, F1, per = [], [], [], {}
    for c in (0, 1):
        tp = sum(1 for g, p in zip(gold, pred) if g == c and p == c)
        npred = sum(1 for p in pred if p == c); ngold = sum(1 for g in gold if g == c)
        p = tp / npred if npred else 0.0; r = tp / ngold if ngold else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        P.append(p); R.append(r); F1.append(f); per[c] = f
    acc = sum(1 for g, p in zip(gold, pred) if g == p) / max(1, len(gold))
    return dict(macro_f1=sum(F1) / 2, macro_p=sum(P) / 2, macro_r=sum(R) / 2, f1_class0=per[0], f1_class1=per[1], acc=acc)

def best_threshold(probs, gold, grid=None):
    """Post-processing: decision threshold on P(class 1) that maximises dev macro-F1 (a calibration step - report it as one)."""
    grid = grid or [i / 100 for i in range(5, 96, 1)]
    best = max(grid, key=lambda t: binary_scores(gold, [int(p >= t) for p in probs])["macro_f1"])
    return best, binary_scores(gold, [int(p >= best) for p in probs])["macro_f1"]

## 6. Adaptation methods: LoRA, LoRMA+, bottleneck adapters
The 4 unique blocks are run twice, so their adapters are shared across both passes.

In [ ]:
class LoRALinear(nn.Module):
    """LoRA: W x + s * B A x with W frozen.  rs=True -> rank-stabilised scaling s = alpha / sqrt(r) (rsLoRA)."""
    def __init__(self, base, r=16, alpha=32, rs=False, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale = alpha / math.sqrt(r) if rs else alpha / r
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.T @ self.B.T) * self.scale

def apply_lora(module, r=16, alpha=32, rs=False):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            setattr(module, name, LoRALinear(child, r, alpha, rs))
        else:
            apply_lora(child, r, alpha, rs)


class LoRMALinear(nn.Module):
    """LoRMA+ (Bihany, Patel, Modi 2025): multiplicative update  h = (I + (alpha/r) B A) W0 x ,  B=0 at init."""
    def __init__(self, base, r=8, alpha=8):
        super().__init__()
        self.base = base
        d = base.out_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r))
        self.scale = alpha / r
    def forward(self, x):
        z = F.linear(x, self.base.weight)
        z = z + self.scale * ((z @ self.A.T) @ self.B.T)
        return z + self.base.bias if self.base.bias is not None else z

def apply_lorma(module, targets=("q_proj", "v_proj"), r=8, alpha=8):
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, LoRMALinear(child, r, alpha))
        else:
            apply_lorma(child, targets, r, alpha)



class AdapterBlock(nn.Module):
    """Bottleneck adapter (Houlsby-style) after a whole block: x + up(GELU(down(x))). `up` starts at zero = identity."""
    def __init__(self, block, d_model, bottleneck=64):
        super().__init__()
        self.block = block
        self.down = nn.Linear(d_model, bottleneck)
        self.up = nn.Linear(bottleneck, d_model)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        x = self.block(x)
        return x + self.up(F.gelu(self.down(x)))

def apply_adapters(lm, bottleneck=64):
    lm.blocks = nn.ModuleList([AdapterBlock(b, lm.cfg.d_model, bottleneck) for b in lm.blocks])

def n_params(m, trainable=False):
    return sum(p.numel() for p in m.parameters() if (p.requires_grad or not trainable))

## 7. Model, `build_model`, distillation student

In [ ]:
class CJModel(nn.Module):
    """Phase-1 lightweight LM (4 shared blocks x 2 repeats, causal 128-token local attention) reads each chunk -> chunk vector [mean ; last-token state]
    -> document-level aggregation over the chunks (agg = 'attn' learned attention pooling | 'mean' | 'gru' BiGRU)
    -> 2-way outcome logits."""
    def __init__(self, lm, agg="attn", proj=512, rnn_hidden=256, dropout=0.1):
        super().__init__()
        self.lm, self.agg = lm, agg
        d = lm.cfg.d_model
        self.proj = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * d, proj), nn.GELU())
        if agg == "attn":
            self.att = nn.Linear(proj, 1); zdim = proj
        elif agg == "gru":
            self.rnn = nn.GRU(proj, rnn_hidden, batch_first=True, bidirectional=True); zdim = 2 * rnn_hidden
        elif agg == "mean":
            zdim = proj
        else:
            raise ValueError(agg)
        self.drop = nn.Dropout(dropout)
        self.out = nn.Linear(zdim, 2)

    def head_params(self):
        mods = [self.proj, self.out] + [getattr(self, n) for n in ("att", "rnn") if hasattr(self, n)]
        return [p for m in mods for p in m.parameters()]

    def features(self, x):
        h = self.lm.tok_emb(x)
        for _ in range(self.lm.cfg.n_repeats):              # the same unique blocks are run n_repeats times
            for blk in self.lm.blocks:
                h = blk(h)
        return self.lm.ln_f(h)

    def forward(self, X, lens, cnt):
        h = self.features(X)                                                   # (N, T, d)
        N, T, _ = h.shape
        lens = lens.to(h.device)
        m = (torch.arange(T, device=h.device)[None, :] < lens[:, None]).to(h.dtype)
        mean = (h * m.unsqueeze(-1)).sum(1) / lens.unsqueeze(-1).to(h.dtype)
        lastv = h[torch.arange(N, device=h.device), lens - 1]
        v = self.proj(torch.cat([mean, lastv], -1))                            # (N, proj): one vector per chunk
        D, Kmax = cnt.numel(), int(cnt.max())
        pad = v.new_zeros(D, Kmax, v.size(-1))
        mask = torch.zeros(D, Kmax, dtype=torch.bool, device=v.device)
        pos = 0
        for i, c in enumerate(cnt.tolist()):
            pad[i, :c] = v[pos:pos + c]; mask[i, :c] = True; pos += c
        if self.agg == "mean":
            z = (pad * mask.unsqueeze(-1).to(pad.dtype)).sum(1) / mask.sum(1, keepdim=True).to(pad.dtype)
        elif self.agg == "attn":
            s = self.att(pad).squeeze(-1).float().masked_fill(~mask, float("-inf"))
            w = F.softmax(s, dim=1).to(pad.dtype)
            z = (w.unsqueeze(-1) * pad).sum(1)
        else:
            packed = nn.utils.rnn.pack_padded_sequence(pad, cnt, batch_first=True, enforce_sorted=False)
            o, _ = self.rnn(packed)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=Kmax)
            z = (o * mask.unsqueeze(-1).to(o.dtype)).sum(1) / mask.sum(1, keepdim=True).to(o.dtype)
        return self.out(self.drop(z))                                          # (D, 2)


def build_model(method, agg="attn", r=16, alpha=32, lorma_r=8, lorma_alpha=8, bottleneck=64):
    lm = load_lm()
    if method not in ("full", "partial"):
        for p in lm.parameters(): p.requires_grad = False
    if method == "lora":      apply_lora(lm.blocks, r, alpha, rs=False)
    elif method == "lorma":   apply_lorma(lm.blocks, ("q_proj", "v_proj"), lorma_r, lorma_alpha)
    elif method == "adapters": apply_adapters(lm, bottleneck)
    elif method == "partial":                       # layer-wise: freeze embeddings + lower half of the 4 unique blocks (frozen in both passes)
        for p in lm.tok_emb.parameters(): p.requires_grad = False
        for b in lm.blocks[: len(lm.blocks) // 2]:
            for p in b.parameters(): p.requires_grad = False
    elif method != "full":
        raise ValueError(method)
    return CJModel(lm, agg).to(dev)


def make_student(teacher):
    """Smaller student: keep unique blocks 0 and 2 (half the block parameters) and still run them n_repeats times
    (depth 8 -> 4). Shared embeddings, initialised from the teacher incl. its trained head.
    The teacher must be the plain full-fine-tuned model."""
    st = copy.deepcopy(teacher)
    keep = list(range(0, len(st.lm.blocks), 2))
    st.lm.blocks = nn.ModuleList([st.lm.blocks[i] for i in keep])
    st.lm.cfg.n_unique_blocks = len(keep)
    for p in st.parameters(): p.requires_grad = True
    return st

## 8. Inference and evaluation

In [ ]:
@torch.no_grad()
def predict_probs(model, docs, select=None, K=None, bs=4):
    """-> list of P(class 1), aligned with `docs`."""
    model.eval()
    order = sorted(range(len(docs)), key=lambda i: len(docs[i]["chunks"]))
    probs = [0.0] * len(docs)
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        X, lens, cnt, _ = collate([docs[j] for j in idx], select, K)
        with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
            lg = model(X.to(dev), lens, cnt)
        p = F.softmax(lg.float(), -1)[:, 1].cpu().tolist()
        for j, pj in zip(idx, p):
            probs[j] = pj
    return probs

def evaluate(model, docs, select=None, K=None, threshold=0.5, verbose=True):
    probs = predict_probs(model, docs, select, K)
    res = binary_scores([d["label"] for d in docs], [int(p >= threshold) for p in probs])
    res["probs"] = probs
    if verbose:
        print(f"   dev macro-F1 {res['macro_f1']:.4f} (P {res['macro_p']:.4f} R {res['macro_r']:.4f}) | "
              f"F1 class0 {res['f1_class0']:.4f} class1 {res['f1_class1']:.4f} | acc {res['acc']:.4f}")
    return res

## 9. Training loop (all methods; best-epoch restore on dev)

In [ ]:
def train(method, train_docs, dev_docs, epochs=4, bs=2, accum=4, lr_bb=3e-5, lr_head=1e-3, agg="attn",
          teacher=None, kd_T=2.0, kd_alpha=0.5, cw_power=0.5, select=None, K=None, seed=0):
    """method: full | partial | lora | lorma | adapters | kd.   Returns (model restored to its best dev epoch, best dev macro-F1).
    bs = documents per step; each document sends up to K chunks through the model, so lower bs if you run out of memory."""
    random.seed(seed); torch.manual_seed(seed)
    if method == "kd":
        assert teacher is not None, "pass teacher=<your fully fine-tuned model>"
        teacher.eval()
        for p in teacher.parameters(): p.requires_grad = False
        model = make_student(teacher).to(dev)
    else:
        model = build_model(method, agg)
    print(f"[{method}/{agg}/{select or SELECT}] params total {n_params(model)/1e6:.1f}M | trainable {n_params(model, True)/1e6:.2f}M")

    head_ids = {id(p) for p in model.head_params()}
    bb = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    hd = [p for p in model.parameters() if id(p) in head_ids]
    groups, base_lrs = [], []
    if bb: groups.append({"params": bb, "lr": lr_bb}); base_lrs.append(lr_bb)
    groups.append({"params": hd, "lr": lr_head}); base_lrs.append(lr_head)
    opt = torch.optim.AdamW(groups, weight_decay=0.01)

    steps_total = epochs * math.ceil(math.ceil(len(train_docs) / bs) / accum)
    warm = max(1, int(0.06 * steps_total))
    def lr_mult(s):
        return s / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps_total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=dev.type == "cuda")

    cnt = torch.tensor([sum(d["label"] == c for d in train_docs) for c in (0, 1)], dtype=torch.float) + 1
    cw = ((cnt.sum() / cnt / 2) ** cw_power).to(dev) if cw_power else torch.ones(2, device=dev)
    cw = cw / cw.mean()                                          # class-balanced loss (cw_power=0 turns it off)
    print("class weights:", [round(x, 3) for x in cw.tolist()])

    best, best_state, step = -1.0, None, 0
    for ep in range(epochs):
        model.train(); t0 = time.time(); run = 0.0
        order = list(range(len(train_docs))); random.shuffle(order)
        batches = [order[i:i + bs] for i in range(0, len(order), bs)]
        for bi, b in enumerate(batches):
            X, lens, c, Y = collate([train_docs[i] for i in b], select, K)
            X, Y = X.to(dev), Y.to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                lg = model(X, lens, c)
            loss = F.cross_entropy(lg.float(), Y, weight=cw)
            if teacher is not None:
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=dev.type == "cuda"):
                    tl = teacher(X, lens, c)
                kl = F.kl_div(F.log_softmax(lg.float() / kd_T, -1), F.softmax(tl.float() / kd_T, -1),
                              reduction="batchmean") * kd_T ** 2
                loss = kd_alpha * loss + (1 - kd_alpha) * kl
            scaler.scale(loss / accum).backward(); run += loss.item()
            if (bi + 1) % accum == 0 or bi + 1 == len(batches):
                step += 1
                for g, b0 in zip(opt.param_groups, base_lrs): g["lr"] = b0 * lr_mult(step)
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_([p for g in opt.param_groups for p in g["params"]], 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        print(f"ep {ep+1}/{epochs} loss {run/max(1, len(batches)):.4f} ({time.time()-t0:.0f}s)")
        if dev_docs:
            sc = evaluate(model, dev_docs, select, K)
            if sc["macro_f1"] > best:
                best = sc["macro_f1"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state); print(f"restored best epoch, dev macro-F1 {best:.4f}")
    return model, best

## 10. Submission writer
`prediction.json` (a list of `{"id", "pred"}`) goes at the **root** of the zip. Upload each zip to Codabench.

In [ ]:
def make_submission(model, docs, tag, threshold=0.5, select=None, K=None):
    """prediction.json = [{"id": ..., "pred": 0/1}, ...], one entry per test case, zipped with the json at the zip root."""
    probs = predict_probs(model, docs, select, K)
    sub = [{"id": d["id"], "pred": int(p >= threshold)} for d, p in zip(docs, probs)]
    assert len(sub) == len(docs) and all(s["pred"] in (0, 1) for s in sub)
    with open(f"{OUT}/prediction.json", "w", encoding="utf-8") as f:
        json.dump(sub, f)
    name = f"{OUT}/submission_p4_{tag}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(f"{OUT}/prediction.json", arcname="prediction.json")
    print(name, zipfile.ZipFile(name).namelist(), "| cases:", len(sub), "| predicted 1:", sum(s["pred"] for s in sub))
    return name

results = {}

## 11. Experiment 1 - full fine-tuning (also the KD teacher)
Everything trainable (backbone lr 3e-5, head lr 1e-3). Keep this model in memory for distillation. Judgments are long: if you run out of GPU memory, lower `bs` or `K_CHUNKS`.

In [ ]:
m_full, results["full"] = train("full", train_docs, dev_docs)
torch.save(m_full.state_dict(), f"{OUT}/p4_lightweight_full.pt")
make_submission(m_full, test_docs, "lightweight_full")

## 12. Experiment 2 - partial (layer-wise) fine-tuning
Embeddings and the lower half of the 4 unique blocks frozen (in both passes); the upper half and the head train.

In [ ]:
m_part, results["partial"] = train("partial", train_docs, dev_docs)
make_submission(m_part, test_docs, "lightweight_partial")

## 13. Experiment 3 - LoRA
Frozen backbone, rank-16 adapters on every Linear layer (attention and SwiGLU).

In [ ]:
m_lora, results["lora"] = train("lora", train_docs, dev_docs, lr_bb=5e-4)
make_submission(m_lora, test_docs, "lightweight_lora")

## 14. Experiment 4 - LoRMA+ (compulsory)
Multiplicative low-rank adaptation (Bihany, Patel, Modi 2025) of the query/value projections of the attention in each unique block.

In [ ]:
m_lorma, results["lorma"] = train("lorma", train_docs, dev_docs, lr_bb=3e-4)
make_submission(m_lorma, test_docs, "lightweight_lorma")

## 15. Experiment 5 - bottleneck adapters
A small zero-initialised down/up adapter after every unique block; the backbone stays frozen.

In [ ]:
m_adapt, results["adapters"] = train("adapters", train_docs, dev_docs, lr_bb=1e-3)
make_submission(m_adapt, test_docs, "lightweight_adapters")

## 16. Experiment 6 - knowledge distillation
Teacher = experiment 1. Student = teacher keeping unique blocks 0 and 2 (half the block parameters; effective depth 8 -> 4), initialised from the teacher, trained on hard labels + temperature-scaled teacher logits. Compare its parameter count, speed and F1 with the teacher.

In [ ]:
m_kd, results["kd"] = train("kd", train_docs, dev_docs, teacher=m_full, lr_bb=5e-5)
make_submission(m_kd, test_docs, "lightweight_kd")

## 17. Long-document ablations (full fine-tuning; optional, each is a full training run)
Which part of a judgment carries the signal? Compare `head` (beginning), `tail` (end - where the outcome usually is) and `spread` against the default `headtail`; then the aggregator (`attn` is the default, `mean` and `gru` below). Report what the dev scores say about sections.

In [ ]:
for sel in ["head", "tail", "spread"]:
    m_tmp, results[f"full_{sel}"] = train("full", train_docs, dev_docs, select=sel)
    del m_tmp; torch.cuda.empty_cache()
for ag in ["mean", "gru"]:
    m_tmp, results[f"full_agg_{ag}"] = train("full", train_docs, dev_docs, agg=ag)
    del m_tmp; torch.cuda.empty_cache()

## 18. Summary, threshold calibration and error-analysis starters

In [ ]:
print("dev macro-F1:")
for k, v in results.items():
    print(f"  {k:20s} {v:.4f}")

res = evaluate(m_full, dev_docs, verbose=False)
probs, gold = res["probs"], [d["label"] for d in dev_docs]
t, f = best_threshold(probs, gold)
print(f"\nfull fine-tune: macro-F1 at threshold 0.5 = {res['macro_f1']:.4f} | best dev threshold {t:.2f} -> {f:.4f}")
print("   (a tuned threshold is fitted on dev - report it as post-processing, and expect a smaller gain on the test set)")

pred = [int(p >= 0.5) for p in probs]
fp = [d for d, p, g in zip(dev_docs, pred, gold) if p == 1 and g == 0]
fn = [d for d, p, g in zip(dev_docs, pred, gold) if p == 0 and g == 1]
ok = [d for d, p, g in zip(dev_docs, pred, gold) if p == g]
print(f"false positives {len(fp)} | false negatives {len(fn)} | correct {len(ok)}")
for name, grp in [("FP", fp), ("FN", fn), ("correct", ok)]:
    if grp: print(f"  {name:8s} median tokens/doc {int(np.median([d['n_tok'] for d in grp]))}")

## 19. (Optional) retrain on train + dev
Once you know the best method, epoch count and (if used) threshold, retrain on all labelled data (no dev set, so no best-epoch restore) and submit again.
```python
m_final, _ = train("full", train_docs + dev_docs, None, epochs=<best epoch count>)
make_submission(m_final, test_docs, "hybrid_final", threshold=0.5)
```